# P3 - Problema de transporte

Matemáticas avanzadas para la ciencia de datos - 5AV1 - 05/10/2026

**Integrantes:**
- Guzmán Hernández Daniel Diego
-
-

In [21]:
import numpy as np
from scipy.optimize import linprog

## A. Datos

3 origenes (O1, O2, O3) y 4 destinos (D1, D2, D3, D4)

In [22]:
# costos
C = np.array([[2, 9, 4, 7],
              [7, 6, 11, 3],
              [9, 5, 8, 5]])

# oferta y demanda
s = np.array([34, 20, 25])
d = np.array([8, 23, 13, 35])

m = len(s)
n = len(d)
print("m =", m)
print("n =", n)
print("total de variables xij =", m*n)

m = 3
n = 4
total de variables xij = 12


## B. Formulación

$x_{ij}$ = cantidad que se manda del origen $i$ al destino $j$

Función objetivo:

$$\min z = 2x_{11} + 9x_{12} + 4x_{13} + 7x_{14} + 7x_{21} + 6x_{22} + 11x_{23} + 3x_{24} + 9x_{31} + 5x_{32} + 8x_{33} + 5x_{34}$$

Restricciones de oferta:

$x_{11} + x_{12} + x_{13} + x_{14} = 34$

$x_{21} + x_{22} + x_{23} + x_{24} = 20$

$x_{31} + x_{32} + x_{33} + x_{34} = 25$

Restricciones de demanda:

$x_{11} + x_{21} + x_{31} = 8$

$x_{12} + x_{22} + x_{32} = 23$

$x_{13} + x_{23} + x_{33} = 13$

$x_{14} + x_{24} + x_{34} = 35$

No negatividad: $x_{ij} \geq 0$

## C. Balance

In [23]:
print("suma oferta =", s.sum())
print("suma demanda =", d.sum())
print("balanceado?", s.sum() == d.sum())

suma oferta = 79
suma demanda = 79
balanceado? True


Si está balanceado (79 = 79), entonces todas las restricciones son de igualdad y no hay que agregar origen o destino ficticio.

Funciones para las comprobaciones que hay que hacer en cada tabla

In [24]:
def costo(X):
    return np.sum(C * X)

def revisar(X):
    print("filas:", X.sum(axis=1), "oferta:", s, "->", np.all(X.sum(axis=1) == s))
    print("columnas:", X.sum(axis=0), "demanda:", d, "->", np.all(X.sum(axis=0) == d))
    print("no negativos:", np.all(X >= 0))

def potenciales(basicas):
    # u_i + v_j = c_ij en las basicas, con u1 = 0
    u = [None, None, None]
    v = [None, None, None, None]
    u[0] = 0
    while None in u or None in v:
        for (i, j) in basicas:
            if u[i] != None and v[j] == None:
                v[j] = int(C[i][j] - u[i])
            elif v[j] != None and u[i] == None:
                u[i] = int(C[i][j] - v[j])
    return u, v

def costos_reducidos(u, v, basicas):
    # cr_ij = c_ij - u_i - v_j en las NO basicas
    cr = {}
    for i in range(m):
        for j in range(n):
            if (i, j) not in basicas:
                cr[(i, j)] = int(C[i][j] - u[i] - v[j])
                print(f"x{i+1}{j+1}: {C[i][j]} - ({u[i]}) - ({v[j]}) = {cr[(i, j)]}")
    return cr

**Nota:** en python los índices empiezan en 0, entonces la celda (0,0) es $x_{11}$, la (2,1) es $x_{32}$, etc.

## D. SBF inicial - esquina noroeste

Empezamos arriba a la izquierda, asignamos lo máximo posible, si se acaba la oferta bajamos y si se acaba la demanda nos movemos a la derecha.

In [25]:
X = np.zeros((m, n), dtype=int)
of = s.copy()
de = d.copy()
basicas = []

i = 0
j = 0
while i < m and j < n:
    x = min(of[i], de[j])
    X[i][j] = x
    basicas.append((i, j))
    of[i] = of[i] - x
    de[j] = de[j] - x
    if of[i] == 0:
        i = i + 1
    else:
        j = j + 1

print(X)

[[ 8 23  3  0]
 [ 0  0 10 10]
 [ 0  0  0 25]]


In [26]:
print("basicas:", basicas)
print("numero de basicas =", len(basicas))
print("m + n - 1 =", m + n - 1)
revisar(X)
print("costo inicial =", costo(X))

basicas: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (2, 3)]
numero de basicas = 6
m + n - 1 = 6
filas: [34 20 25] oferta: [34 20 25] -> True
columnas: [ 8 23 13 35] demanda: [ 8 23 13 35] -> True
no negativos: True
costo inicial = 500


Basicas: $x_{11}=8,\ x_{12}=23,\ x_{13}=3,\ x_{23}=10,\ x_{24}=10,\ x_{34}=25$

Son 6 = m+n-1 entonces no es degenerada.

Costo $= 2(8) + 9(23) + 4(3) + 11(10) + 3(10) + 5(25) = 500$

## Iteración 1

### Potenciales
Fijamos $u_1 = 0$ y usamos $u_i + v_j = c_{ij}$ en las básicas:

| Celda básica | Ecuación |
|---|---|
| $x_{11}$ | $u_1 + v_1 = 2$ |
| $x_{12}$ | $u_1 + v_2 = 9$ |
| $x_{13}$ | $u_1 + v_3 = 4$ |
| $x_{23}$ | $u_2 + v_3 = 11$ |
| $x_{24}$ | $u_2 + v_4 = 3$ |
| $x_{34}$ | $u_3 + v_4 = 5$ |

In [27]:
u, v = potenciales(basicas)
print("u =", u)
print("v =", v)

u = [0, 7, 9]
v = [2, 9, 4, -4]


### Costos reducidos
$\bar c_{ij} = c_{ij} - u_i - v_j$

In [28]:
cr = costos_reducidos(u, v, basicas)

x14: 7 - (0) - (-4) = 11
x21: 7 - (7) - (2) = -2
x22: 6 - (7) - (9) = -10
x31: 9 - (9) - (2) = -2
x32: 5 - (9) - (9) = -13
x33: 8 - (9) - (4) = -5


Hay varios negativos ($x_{21}, x_{22}, x_{31}, x_{32}, x_{33}$), entonces todavía se puede mejorar.
Escogemos **$x_{32}$** porque tiene el más negativo ($-13$), es la que más baja el costo por cada unidad.

In [29]:
entra = min(cr, key=cr.get)
print("entra:", entra, "con cr =", cr[entra])

entra: (2, 1) con cr = -13


### Ciclo
Desde $x_{32}$ buscamos un ciclo cerrado que solo pase por básicas, moviéndonos por fila y columna:

$x_{32}(+) \to x_{34}(-) \to x_{24}(+) \to x_{23}(-) \to x_{13}(+) \to x_{12}(-)$

$\theta = \min\{x_{34}, x_{23}, x_{12}\} = \min\{25, 10, 23\} = 10$

Sale $x_{23}$

In [30]:
ciclo = [(2,1), (2,3), (1,3), (1,2), (0,2), (0,1)]   # x32, x34, x24, x23, x13, x12
signos = [+1, -1, +1, -1, +1, -1]

menos = [int(X[c]) for c, sg in zip(ciclo, signos) if sg == -1]
print("celdas con signo - :", menos)
theta = min(menos)
print("theta =", theta)

for c, sg in zip(ciclo, signos):
    X[c] = X[c] + sg*theta

basicas.remove((1,2))   # sale x23
basicas.append((2,1))   # entra x32

print(X)
revisar(X)
print("costo =", costo(X))

celdas con signo - : [25, 10, 23]
theta = 10
[[ 8 13 13  0]
 [ 0  0  0 20]
 [ 0 10  0 15]]
filas: [34 20 25] oferta: [34 20 25] -> True
columnas: [ 8 23 13 35] demanda: [ 8 23 13 35] -> True
no negativos: True
costo = 370


El costo bajó de 500 a 370 (bajó $13 \times 10 = 130$)

## Iteración 2

Básicas: $x_{11}, x_{12}, x_{13}, x_{24}, x_{32}, x_{34}$

### Potenciales

In [31]:
u, v = potenciales(basicas)
print("u =", u)
print("v =", v)

u = [0, -6, -4]
v = [2, 9, 4, 9]


### Costos reducidos

In [32]:
cr = costos_reducidos(u, v, basicas)
entra = min(cr, key=cr.get)
print("entra:", entra, "con cr =", cr[entra])

x14: 7 - (0) - (9) = -2
x21: 7 - (-6) - (2) = 11
x22: 6 - (-6) - (9) = 3
x23: 11 - (-6) - (4) = 13
x31: 9 - (-4) - (2) = 11
x33: 8 - (-4) - (4) = 8
entra: (0, 3) con cr = -2


Solo $x_{14}$ es negativo ($-2$), entonces **entra $x_{14}$**.

### Ciclo
$x_{14}(+) \to x_{12}(-) \to x_{32}(+) \to x_{34}(-)$

$\theta = \min\{13, 15\} = 13$

Sale $x_{12}$

In [33]:
ciclo = [(0,3), (0,1), (2,1), (2,3)]   # x14, x12, x32, x34
signos = [+1, -1, +1, -1]

menos = [int(X[c]) for c, sg in zip(ciclo, signos) if sg == -1]
print("celdas con signo - :", menos)
theta = min(menos)
print("theta =", theta)

for c, sg in zip(ciclo, signos):
    X[c] = X[c] + sg*theta

basicas.remove((0,1))   # sale x12
basicas.append((0,3))   # entra x14

print(X)
revisar(X)
print("costo =", costo(X))

celdas con signo - : [13, 15]
theta = 13
[[ 8  0 13 13]
 [ 0  0  0 20]
 [ 0 23  0  2]]
filas: [34 20 25] oferta: [34 20 25] -> True
columnas: [ 8 23 13 35] demanda: [ 8 23 13 35] -> True
no negativos: True
costo = 344


El costo bajó de 370 a 344 (bajó $2 \times 13 = 26$)

## Iteración 3 - verificar optimalidad

Básicas: $x_{11}, x_{13}, x_{14}, x_{24}, x_{32}, x_{34}$

In [34]:
u, v = potenciales(basicas)
print("u =", u)
print("v =", v)
print()
cr = costos_reducidos(u, v, basicas)
print()
print("todos >= 0?", all(c >= 0 for c in cr.values()))

u = [0, -4, -2]
v = [2, 7, 4, 7]

x12: 9 - (0) - (7) = 2
x21: 7 - (-4) - (2) = 9
x22: 6 - (-4) - (7) = 3
x23: 11 - (-4) - (4) = 11
x31: 9 - (-2) - (2) = 9
x33: 8 - (-2) - (4) = 6

todos >= 0? True


Todos los $\bar c_{ij} \geq 0$ en las celdas no básicas, entonces **la solución es óptima**.
(Además ninguno es 0, entonces no hay otra solución con el mismo costo.)

## Solución final (método de transporte)

In [35]:
X_final = X.copy()
print(X_final)
revisar(X_final)
z_final = costo(X_final)
print("costo minimo =", z_final)

[[ 8  0 13 13]
 [ 0  0  0 20]
 [ 0 23  0  2]]
filas: [34 20 25] oferta: [34 20 25] -> True
columnas: [ 8 23 13 35] demanda: [ 8 23 13 35] -> True
no negativos: True
costo minimo = 344


$x_{11}=8,\ x_{13}=13,\ x_{14}=13,\ x_{24}=20,\ x_{32}=23,\ x_{34}=2$, lo demás es 0

$z = 2(8) + 4(13) + 7(13) + 3(20) + 5(23) + 5(2) = 344$

## I. Verificación con linprog

Ordenamos las 12 variables por filas:
$(x_{11}, x_{12}, x_{13}, x_{14}, x_{21}, x_{22}, x_{23}, x_{24}, x_{31}, x_{32}, x_{33}, x_{34})$

Las restricciones de oferta y demanda van como igualdades y bounds $x \geq 0$

In [36]:
c = C.flatten()

A_eq = [
    [1,1,1,1, 0,0,0,0, 0,0,0,0],   # oferta O1
    [0,0,0,0, 1,1,1,1, 0,0,0,0],   # oferta O2
    [0,0,0,0, 0,0,0,0, 1,1,1,1],   # oferta O3
    [1,0,0,0, 1,0,0,0, 1,0,0,0],   # demanda D1
    [0,1,0,0, 0,1,0,0, 0,1,0,0],   # demanda D2
    [0,0,1,0, 0,0,1,0, 0,0,1,0],   # demanda D3
    [0,0,0,1, 0,0,0,1, 0,0,0,1],   # demanda D4
]
b_eq = [34, 20, 25, 8, 23, 13, 35]

res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=(0, None))
print(res.message)
X_solver = res.x.reshape(3, 4)
print(X_solver)
print("costo solver =", res.fun)

Optimization terminated successfully. (HiGHS Status 7: Optimal)
[[ 8.  0. 13. 13.]
 [ 0.  0.  0. 20.]
 [ 0. 23.  0.  2.]]
costo solver = 344.0


## Comparación

In [37]:
print("costo metodo =", z_final)
print("costo linprog =", res.fun)
print("mismo costo?", np.isclose(z_final, res.fun))
print("misma solucion?", np.allclose(X_final, X_solver))

costo metodo = 344
costo linprog = 344.0
mismo costo? True
misma solucion? True


Salen iguales, el solver confirma que el costo mínimo es 344 con la misma asignación.

## J. Interpretación

El plan de envío más barato es:

- O1 manda 8 a D1, 13 a D3 y 13 a D4
- O2 manda sus 20 a D4
- O3 manda 23 a D2 y 2 a D4

Con esto se cubre exacto la oferta y la demanda y el costo mínimo es **344**.

Tiene sentido porque cada origen usa sus rutas más baratas: O1 manda a D1 y D3 (costos 2 y 4), O2 todo a D4 (costo 3) y O3 cubre D2 (costo 5). D4 es el que más pide (35) por eso recibe de los tres.

La esquina noroeste empezó en 500 porque no toma en cuenta los costos (mandaba 23 por O1→D2 que cuesta 9). Con el método se bajó a 370 y luego a 344.